# Threat hunting playbook

Hypothesis-driven hunt on **labeled network-flow telemetry** and **published IOCs**. This example **registers CICIDS2017 as a Cribl Search dataset** (Generic HTTP API provider via REST), **`join`s the dataset with an IOC lookup**, **interprets** the enriched results, and **cleans up** when finished.

**Telemetry:** [CICIDS2017](https://unb.ca/cic/datasets/ids-2017.html) sample (Western-OC2-Lab mirror). **Intel:** [SophosLabs IoCs](https://github.com/sophoslabs/IoCs) (EDR-killer campaign, Aug 2025).

## Hunt charter

**Hypothesis:** Attack-labeled flows in the CICIDS sample cluster by `Label`, and published hash IOCs can be loaded into Search for enrichment-style hunting on tenants that expose file-hash fields.

**Questions:**

1. Can we register the public CICIDS CSV as a Search **dataset** (not only `externaldata`)?
2. Which attack labels dominate non-`BENIGN` traffic?
3. Can we **`join`** attack flows from the CICIDS **dataset** with the IOC **lookup** and interpret the results?

**Scope:** Demo IDs `notebook_app_threat_hunt_http` (provider), `notebook_app_cicids2017` (dataset), and lookup `notebook_app_threat_hunt_iocs.csv` — all removed in **Cleanup**.

## Prerequisites

1. Run inside a hosted Cribl app where **Cribl Search** and **`%%cribl_api`** work (`default_search` group).
2. Search must reach `raw.githubusercontent.com` (dataset provider fetches the CSV at query time).
3. Lookups are capped at **10,000 rows**; this playbook uses ~50 IOC rows.

**Related:** `Cribl_Search_Lookup_Magics.ipynb` (lookup magics), `Cribl_API_Examples.ipynb` (Search jobs API), `Incident_Triage_Playbook.ipynb` (live-sample triage).

### A) Configure CICIDS2017 — Generic HTTP API dataset provider

Uses Search REST API paths under `/m/default_search/search/dataset-providers` ([Generic HTTP API](https://docs.cribl.io/search/set-up-generic-http-api/)).

Re-running this notebook: **Cleanup** at the end deletes these objects; if a prior run stopped early, run the cleanup cells first or delete the same IDs in the Search UI.

In [ ]:
%%cribl_api DELETE /m/default_search/search/datasets/notebook_app_cicids2017 response=json


In [ ]:
%%cribl_api DELETE /m/default_search/search/dataset-providers/notebook_app_threat_hunt_http response=json


In [ ]:
%%cribl_api POST /m/default_search/search/dataset-providers var=threat_http_provider response=json
json:
  type: api_http
  id: notebook_app_threat_hunt_http
  description: Notebook threat hunt — CICIDS2017 CSV (Generic HTTP API)
  authenticationMethod: none
  availableEndpoints:
    - name: cicids_csv
      method: GET
      url: https://raw.githubusercontent.com/Western-OC2-Lab/Intrusion-Detection-System-Using-Machine-Learning/main/data/CICIDS2017_sample.csv
      headers: []
      dataField: ""


### B) Register the CICIDS2017 Search dataset

Federated **api_http** dataset with **`breakerRulesets`** so the CSV is parsed into columns (not one giant JSON blob). Without `CSV Datatypes`, searches may fail with API read errors.

In [ ]:
%%cribl_api POST /m/default_search/search/datasets var=threat_cicids_dataset response=json
json:
  type: api_http
  id: notebook_app_cicids2017
  description: CICIDS2017 sample flows (notebook threat hunt)
  provider: notebook_app_threat_hunt_http
  enabledEndpoints:
    - cicids_csv
  filter: "true"
  searchVersion: v1
  breakerRulesets:
    - CSV Datatypes
    - Cribl Search
  metadata:
    enableAcceleration: false


### C) Load CICIDS dataset (source ground truth)

1. **`flows_label_stats`** — lightweight `summarize` (always completes quickly).
2. **`flows_df`** — full dataset (`limit=0`); the app paginates `/results` in small pages with retries and loads Pyodide in chunks.

Counts should match [Western-OC2-Lab CICIDS2017_sample.csv](https://github.com/Western-OC2-Lab/Intrusion-Detection-System-Using-Machine-Learning/blob/main/data/CICIDS2017_sample.csv) (~56,661 rows: ~22,731 `BENIGN`, ~33,930 attack-labeled).

In [ ]:
%%cribl_search var=flows_label_stats lang=kql limit=0 preview=true earliest=-50y latest=now
dataset="notebook_app_cicids2017"
| summarize flow_count=count() by Label
| sort by flow_count desc

In [ ]:
%%cribl_search var=flows_df lang=kql limit=0 preview=false earliest=-50y latest=now
dataset="notebook_app_cicids2017"

In [ ]:
import pandas as pd

def _pick_col(frame, *names):
    keys = {str(c).strip().lower(): c for c in frame.columns}
    for n in names:
        if n in keys:
            return keys[n]
    return None

stats_label_col = _pick_col(flows_label_stats, 'label')
stats_count_col = _pick_col(flows_label_stats, 'flow_count', 'count')
label_col = _pick_col(flows_df, 'label') or stats_label_col

if stats_label_col and stats_count_col:
    _stats = flows_label_stats.copy()
    _stats['_label'] = _stats[stats_label_col].astype(str).str.strip()
    _stats['_n'] = pd.to_numeric(_stats[stats_count_col], errors='coerce').fillna(0).astype(int)
    source_total = int(_stats['_n'].sum())
    source_benign = int(_stats.loc[_stats['_label'].str.upper() == 'BENIGN', '_n'].sum())
    source_attack = source_total - source_benign
    source_attack_counts = (
        _stats.loc[_stats['_label'].str.upper() != 'BENIGN', ['_label', '_n']]
        .set_index('_label')['_n']
        .sort_values(ascending=False)
    )
    print('Label stats from Search summarize (source-aligned):')
    print(_stats[[stats_label_col, stats_count_col]].to_string(index=False))
else:
    source_attack_counts = pd.Series(dtype=int)
    source_total = source_benign = source_attack = 0

print()
print('flows rows loaded (dataset):', len(flows_df))
print('columns (first 12):', list(flows_df.columns)[:12], '…' if len(flows_df.columns) > 12 else '')
print(f'source alignment: total={source_total}, BENIGN={source_benign}, attack-labeled={source_attack}')
print('(Expect ~56661 total, ~22731 BENIGN, ~33930 attack)')

if label_col and len(flows_df):
    loaded_labels = flows_df[label_col].astype(str).str.strip()
    loaded_total = len(flows_df)
    if loaded_total != source_total and source_total:
        print(f'note: loaded {loaded_total} rows vs summarize total {source_total} (pagination cap or job still materializing)')
    flows_hunt_df = flows_df[loaded_labels.str.upper() != 'BENIGN'].copy()
else:
    flows_hunt_df = flows_df.copy()

flows_hunt_df.head(5)

### D) Load IOCs and publish a Search lookup

Small IOC table via `externaldata` (one-shot intel fetch), then **`%%cribl_save_search_lookup`** for `$vt_lookups` / join-adjacent queries. On production data, replace this with your TI feed or lookup pipeline.

In [ ]:
%%cribl_search var=ioc_raw lang=kql limit=0 preview=false earliest=-50y latest=now
externaldata
[
  "https://raw.githubusercontent.com/sophoslabs/IoCs/master/06082025-edrkiller-iocs.csv"
]
with(
  datatype="CSV Datatypes"
)

In [ ]:
def _pick_col(frame, *names):
    lower = {str(c).strip().lower(): c for c in frame.columns}
    for n in names:
        if n in lower:
            return lower[n]
    return None

tcol = _pick_col(ioc_raw, 'indicator_type')
dcol = _pick_col(ioc_raw, 'data')
ncol = _pick_col(ioc_raw, 'note')

if tcol and dcol:
    ioc_lookup_df = pd.DataFrame({
        'indicator_type': ioc_raw[tcol].astype(str),
        'value': ioc_raw[dcol].astype(str),
        'note': ioc_raw[ncol].astype(str) if ncol else '',
    })
    ioc_lookup_df = ioc_lookup_df[
        ioc_lookup_df['indicator_type'].str.lower() != 'description'
    ].reset_index(drop=True)
else:
    cols = list(ioc_raw.columns)[:3]
    ioc_lookup_df = ioc_raw[cols].copy()
    ioc_lookup_df.columns = ['indicator_type', 'value', 'note'][: len(cols)]

# Shared join key: correlate this hunt's flows with the IOC lookup (production: use file_hash, src_ip, etc.)
HUNT_CAMPAIGN = 'edr_killer_2025'
ioc_lookup_df['hunt_campaign'] = HUNT_CAMPAIGN

ioc_sha_df = ioc_lookup_df[ioc_lookup_df['indicator_type'].str.lower() == 'sha256'].copy()
source_ioc_sha_count = len(ioc_sha_df)
print('IOC rows for lookup:', len(ioc_lookup_df))
print(f'SHA-256 IOCs (source): {source_ioc_sha_count} (expect 51 — Sophos 06082025-edrkiller-iocs.csv)')
print('IOC notes (source):')
print(ioc_sha_df['note'].value_counts().head(8).to_string())
ioc_lookup_df.head(10)

In [ ]:
%%cribl_save_search_lookup notebook_app_threat_hunt_iocs.csv var=ioc_lookup_df replace=true


### E) Threat hunt — `join` CICIDS dataset with IOC lookup

**Left scope:** non-`BENIGN` flows from `notebook_app_cicids2017`, tagged with `hunt_campaign`.

**Right scope:** SHA-256 rows from `$vt_lookups` / `notebook_app_threat_hunt_iocs` (same `hunt_campaign`).

**Join:** `leftouter` on `hunt_campaign` so each attack flow is enriched with every campaign SHA-256 IOC (full cartesian enrichment). No `| limit` on the pipeline; **`limit=0`** on the magic loads all result pages.

The CICIDS sample has no `file_hash` column, so we use a **campaign scope key** for this demo. Expected join rows ≈ **attack-labeled flows × 51 SHA-256 IOCs** from the source files. On EDR telemetry, join on `$left.file_hash == $right.value` instead.

In [ ]:
%%cribl_search var=hunt_enrich_df lang=kql limit=0 preview=true earliest=-50y latest=now template=on
dataset="notebook_app_cicids2017"
| where Label != "BENIGN"
| extend hunt_campaign = "{{ HUNT_CAMPAIGN }}"
| join kind=leftouter (
    dataset="$vt_lookups" lookupFile="notebook_app_threat_hunt_iocs"
    | where indicator_type =~ "sha256"
    | extend hunt_campaign = "{{ HUNT_CAMPAIGN }}"
    | project hunt_campaign, indicator_type, ioc_value=value, ioc_note=note
  ) on $left.hunt_campaign == $right.hunt_campaign

### F) Interpret findings

Compare **join output** to **source ground truth** from §C (full CICIDS dataset) and §D (Sophos IOC CSV). Attack-label mix and row counts should line up with the public files; the join row count should approximate **attack flows × SHA-256 IOCs** (campaign-scoped enrichment, not hash detection on flows).

In [ ]:
import matplotlib.pyplot as plt

join_label_col = _pick_col(hunt_enrich_df, 'label')
ioc_col = _pick_col(hunt_enrich_df, 'ioc_value', 'value')
note_col = _pick_col(hunt_enrich_df, 'ioc_note', 'note')
dur_col = _pick_col(hunt_enrich_df, 'flow duration')

n_join = len(hunt_enrich_df)
expected_join = (
    int(source_attack) * int(source_ioc_sha_count)
    if label_col is not None and 'source_attack' in globals()
    else None
)

print('=== Source ground truth (public files) ===')
if label_col is not None and 'source_attack' in globals():
    print(f'CICIDS dataset rows: {source_total} (expect ~56661)')
    print(f'  BENIGN: {source_benign} (expect ~22731)')
    print(f'  Attack-labeled: {source_attack} (expect ~33930)')
    print('  Top attack labels:')
    print(source_attack_counts.head(6).to_string())
print(f'Sophos SHA-256 IOCs: {source_ioc_sha_count} (expect 51)')
if expected_join:
    print(f'Expected full join rows (attack × SHA-256): {expected_join:,}')
print()

print('=== Join output ===')
print(f'Rows returned: {n_join:,}')
if expected_join and n_join > 0:
    ratio = n_join / expected_join
    print(f'Fraction of expected cartesian enrichment: {ratio:.4f}')
    if n_join < expected_join:
        print('  (Search may cap maxResultsPerSearch on very large jobs — compare label mix below.)')
print()

if n_join == 0:
    print('Verdict: INCONCLUSIVE — no join rows. Complete §A–D first.')
elif join_label_col is None or label_col is None:
    print('Verdict: INCONCLUSIVE — missing Label column.')
else:
    join_labels = hunt_enrich_df[join_label_col].astype(str).str.strip()
    join_flow_counts = join_labels.value_counts()
    # Distinct flows per label in join (dedupe IOC expansion)
    if dur_col:
        fk = join_labels + '|' + hunt_enrich_df[dur_col].astype(str)
        join_unique_by_label = hunt_enrich_df.assign(_fk=fk).groupby(join_labels)['_fk'].nunique()
    else:
        join_unique_by_label = join_flow_counts

    align = source_attack_counts.align(join_unique_by_label, join='outer', fill_value=0)
    compare = pd.DataFrame({
        'source_flows': align[0],
        'join_unique_flows': align[1],
    }).astype(int)
    compare['delta'] = compare['join_unique_flows'] - compare['source_flows']
    mismatch = compare[compare['delta'] != 0]

    print('Attack-label alignment (unique flows per label: source vs join-deduped):')
    print(compare.to_string())
    print()

    if len(mismatch) == 0 and n_join >= min(expected_join, 1):
        print(
            'Verdict: ALIGNED — join dedupe matches CICIDS source label counts; IOC inventory matches Sophos SHA-256 list. '
            'Campaign join enriched attack flows with all 51 EDR-killer hashes (analyst bundle, not per-flow hash hits).'
        )
    elif len(mismatch) == 0:
        print('Verdict: PARTIAL — label counts align but join row total below expected (likely platform result cap).')
    else:
        print('Verdict: MISMATCH — review label parsing or incomplete dataset/lookup load.')
        print(mismatch.to_string())

    print()
    print('Source findings (defender summary):')
    top = source_attack_counts.head(3)
    for lab, cnt in top.items():
        pct = 100.0 * cnt / source_attack
        print(f'  - {lab}: {cnt:,} flows ({pct:.1f}% of attack traffic)')
    print(f'  - Sophos EDR-killer IOCs: {source_ioc_sha_count} SHA-256 hashes for campaign correlation')
    if note_col and ioc_col:
        print('  - Sample IOC notes:', ', '.join(ioc_sha_df['note'].dropna().unique()[:4]))

if label_col is not None and 'source_attack_counts' in globals() and len(source_attack_counts):
    ax = source_attack_counts.head(8).plot(
        kind='bar',
        title='CICIDS source: attack-labeled flows by Label',
        rot=45,
        color='steelblue',
    )
    ax.set_xlabel('Label')
    ax.set_ylabel('Flow count (source)')
    plt.tight_layout()
    plt.show()

### Production bridge

- Keep **dataset provider + dataset** definitions in Git ([Cribl as Code](https://docs.cribl.io/cribl-as-code/api/)) instead of notebook-only REST calls when IDs are long-lived.
- Point `dataset="your_index"` at live telemetry; **`join`** or **`| lookup`** on real keys (`file_hash`, `srcaddr`, `user`) instead of `hunt_campaign`.
- Delete ephemeral demo objects (below) so IDs do not collide across classes or tenants.

In [ ]:
# ### Prompt:
# Draft a short threat-hunt summary from the notebook context below.
#
# Dataset + lookup join sample:
# {{ hunt_enrich_df | describe }}
#
# Requirements:
# - interpret campaign-key join vs hash-level detection
# - list top attack labels and IOC enrichment takeaway
# - recommend one production join key for the tenant

### Cleanup

Removes the demo **dataset**, **dataset provider**, and **lookup** created in this notebook.

In [ ]:
%%cribl_delete_search_lookup notebook_app_threat_hunt_iocs.csv


In [ ]:
%%cribl_api DELETE /m/default_search/search/datasets/notebook_app_cicids2017 response=json


In [ ]:
%%cribl_api DELETE /m/default_search/search/dataset-providers/notebook_app_threat_hunt_http response=json


## Debrief

- **Dataset REST API:** `Cribl_API_Examples.ipynb`, Search UI → Data → Dataset Providers / Datasets
- **Lookups:** `Cribl_Search_Lookup_Magics.ipynb`
- **Live-sample triage:** `Incident_Triage_Playbook.ipynb`

**Attribution:** CICIDS2017 ([UNB CIC](https://unb.ca/cic/datasets/ids-2017.html)); IOCs ([SophosLabs IoCs](https://github.com/sophoslabs/IoCs), EDR-killer report Aug 2025).